# E1 · Detección Hα

**Spec:** [`docs/spec_E1_codex_halpha_detection.md`](../docs/spec_E1_codex_halpha_detection.md)  |  **Bloque:** E · Resultado  |  **Run de este set:** `ROXs42Bb_realigned`

Test de detección de emisión Hα del compañero con matched filter y controles.

| | |
|---|---|
| **Entrada** | `spec_final_object.fits` + controles |
| **Salida (QC/productos)** | `stages/stage_h01_qc.json` |
| **Consume aguas abajo** | E2, E3, G2 (V3) |


## Qué hace E1 y el resultado

E1 busca **emisión de Hα** del compañero con un **matched filter** (plantilla de la línea esperada) y calibra la significancia con **controles** (FAP empírico). Es el **endpoint científico**.

Por método: busca en ±500 km/s alrededor de Hα (6562.8 Å, rv_sys -7) → un `z` del matched filter. La **FAP** = fracción de los 33 máximos nulos (posiciones de control) que superan el pico del objeto. **Criterio de detección:** `global_fap < 0.01` **Y** un par admisible (psffit+aperture) **Y** rv dentro de la LSF.

**VEREDICTO = `non_detection`** (`no_method_passes_global_fap`). Valores de **este objeto**: los picos van de z 0.15–2.63 con FAP 0.03–1.00. El pico más alto es **aperture** (z=2.63, FAP=0.03, v=+18 km/s, rv_ok=True, FWHM=2.7 Å).

> **Cómo leer un pico alto:** un `z` grande no es señal por sí solo. Hay que exigirle las tres cosas del criterio — FAP baja frente a SUS controles, rv consistente con el sistémico, y anchura compatible con la LSF. Un pico con rv incoherente o mucho más ancho que la LSF es ruido de borde, no una línea. La celda de evidencia y el Plot 1 dan las tres por método.

**Inputs:** LSF 2.297 Å, rv_sys -7 km/s (estimación de literatura), 33 controles → `min_resolvable_fap` ≈ 0.029 (por encima del umbral: un FAP<1% estricto necesitaría ~99 controles — salvedad).


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_h01_detect.sh --run-id $RUN
```

Ligero–moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_h01_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_h01_detect.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_h01_qc.json', RUN_ID)
nb.show(qc, keys=['verdict', 'reason', 'global_fap_lt', 'significant_methods', 'rv_consistent'], title='E1')


## Los chequeos del QC, en físico

E1 es el endpoint: aquí se decide si hay o no emisión. Los chequeos existen para que un «sí» no pueda venir de un control contaminado ni de una FAP mal calibrada.

| Chequeo | ¿Qué pregunta contesta? | Si falla |
|---|---|---|
| `v2_controls` | **¿La distribución de máximos nulos es sana?** Sin bimodalidades ni outliers extremos que delaten un control contaminado por otra fuente. | La FAP se calibra contra una distribución sucia: el umbral de detección deja de valer. (Se puede excluir un control contaminado, con registro, y recalcular — máximo 2.) |
| `v3_placebo` | **¿La cadena «detecta» donde no puede haber nada?** La misma maquinaria, centrada en 6400 y 6700 Å (líneas placebo), no debe superar el umbral. | Si un placebo detecta, **la FAP está mal calibrada** y la detección de Hα no vale: es hallazgo mayor, no un detalle. La batería completa de placebos es E2/T5. |
| `v4_multimethod` | **¿Los métodos coinciden?** Las FAP por método, contadas y comparadas. | Una discrepancia grande entre métodos apunta a que el resultado depende de cómo se restó el halo → insumo directo para E2. |


## Resultados que llevaron a la conclusión

Veredicto, criterio y el pico/FAP/rv de cada método del `stage_h01_qc.json` + la tabla.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('E1', 'stages/stage_h01_qc.json'):
        import pandas as pd
        q = nb.load_qc('stages/stage_h01_qc.json', RUN_ID)
        v = q['verdict']; cr = q['criterion']
        print('VEREDICTO:', v['verdict'], '|', v['reason'], '| métodos significativos:', v['significant_methods'])
        print(f"criterio: global_fap<{cr['global_fap_lt']}, par admisible {cr['admissible_pairs']}, rv dentro de LSF")
        print(f"línea: Hα {q['line']['rest_A']} Å, rv_sys {q['line']['rv_sys_kms']} km/s, búsqueda ±{q['line']['search_half_width_kms']:.0f} km/s")
        print()
        d = pd.read_csv(nb.run_dir(RUN_ID) / 'tables' / 'halpha_detection_by_method.csv')
        for _, r in d.iterrows():
            print(f"  {r['method']:15s} z={r['matched_z']:.2f}  FAP={r['global_empirical_fap']:.2f}  "
                  f"v={r['peak_velocity_kms']:+.0f} km/s  rv_ok={r['rv_consistent']}  fwhm={r['fwhm_A']:.1f} Å")
        n_ctrl = int(d['n_controls'].iloc[0])
        n_need = int(round(1 / cr['global_fap_lt'])) - 1   # FAP mínimo resoluble = 1/(n+1)
        print(f"\nmin_resolvable_fap = {d['minimum_resolvable_fap'].iloc[0]:.3f} "
              f"({n_ctrl} controles; <{cr['global_fap_lt']} necesita ~{n_need})")


## Plot 1 — el pico del objeto frente a su distribución nula

Por método, los **máximos nulos** (matched filter en las posiciones de control, gris) y el **pico del objeto** (estrella; roja si `rv_consistent=False`). La lectura es la posición del pico **dentro de su propia nube**: si queda inmerso en ella, la FAP es alta y no hay detección. Para este objeto el veredicto es `non_detection` con FAP 0.03–1.00 frente al umbral 0.01.


In [ ]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    rd = nb.run_dir(RUN_ID)
    z = np.load(rd / 'stages' / 'stage_h01_null_maxima.npz')
    d = pd.read_csv(rd / 'tables' / 'halpha_detection_by_method.csv').set_index('method')
    methods = ['aperture', 'optimal_psfsub', 'psffit', 'optimal_ls']
    rng = np.random.default_rng(1)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i, m in enumerate(methods):
        nulls = z[f'{m}_null_maxima']; x = i + rng.uniform(-0.12, 0.12, nulls.size)
        ax.scatter(x, nulls, s=14, color='0.6', alpha=0.7, label=f'máximos nulos ({nulls.size} controles)' if i == 0 else None)
        obj = d.loc[m, 'matched_z']; fap = d.loc[m, 'global_empirical_fap']; rv = d.loc[m, 'rv_consistent']
        ax.scatter(i, obj, s=170, marker='*', color='tab:orange' if rv else 'tab:red', zorder=5,
                   edgecolor='k', label='pico del objeto' if i == 0 else None)
        ax.text(i, obj + 0.35, f'z={obj:.2f}\nFAP={fap:.2f}\nrv_ok={rv}', ha='center', fontsize=7)
    ax.set_xticks(range(len(methods))); ax.set_xticklabels(methods, fontsize=9)
    ax.set_ylabel('z del matched filter (máximo en la ventana Hα)')
    v = nb.load_qc('stages/stage_h01_qc.json', RUN_ID)['verdict']
    ax.set_title(f"E1 · {v['verdict']} ({v['reason']}): pico del objeto vs su nube nula")
    ax.legend(fontsize=8, loc='upper left'); fig.tight_layout()
    outdir = rd / 'plots' / 'e1_halpha'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'detection.png', dpi=110); print('figura ->', outdir / 'detection.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — la región de Hα en el espectro canónico

El espectro canónico (`spec_final_object.fits`) alrededor de Hα con la banda ±1σ empírica (controles del método canónico de D2) y la posición esperada de Hα (rest_A del QC de E1, corrida por rv_sys). Lo que hay que mirar: si asoma algo **por encima de la banda** justo en la posición esperada. Veredicto de E1 para este objeto: `non_detection`.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    from musepipe.constants import C_KMS
    rd = nb.run_dir(RUN_ID)
    line = nb.load_qc('stages/stage_h01_qc.json', RUN_ID)['line']   # rest_A + rv_sys oficiales
    canon = nb.load_qc('stages/stage_x11_qc.json', RUN_ID)['canonical_method']
    h = fits.open(rd / 'stages' / 'spec_final_object.fits')
    wave = np.asarray(h[1].data['wave_A'], float)   # eje λ del propio producto
    flux = np.asarray(h[1].data['flux'], float); h.close()
    sig = np.nanstd(np.load(rd / 'stages' / f'spec_calibrated_{canon}_controls.npz')['control_spectra'], axis=0)
    ha = line['rest_A'] * (1 + line['rv_sys_kms'] / C_KMS)
    w = (wave >= 6400) & (wave <= 6750)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.fill_between(wave[w], -sig[w], sig[w], color='0.85', label='±1σ empírico')
    ax.plot(wave[w], flux[w], lw=0.9, color='tab:blue', label=f'flujo {canon}')
    ax.axvline(ha, color='tab:red', ls=':', label=f'Hα esperado ({ha:.1f} Å)')
    ax.axhline(0, color='0.6', lw=0.6)
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('flujo'); ax.legend(fontsize=8)
    ax.set_title('E1 · región de Hα: sin línea sobre el ruido en la posición esperada')
    fig.tight_layout()
    outdir = rd / 'plots' / 'e1_halpha'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'halpha_region.png', dpi=110); print('figura ->', outdir / 'halpha_region.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **VEREDICTO = `non_detection`** (`no_method_passes_global_fap`); métodos significativos: []. FAP por método 0.03–1.00 frente al umbral 0.01. **Es el endpoint científico de la cadena.**
- Un `z` alto no basta: el pico mayor de este objeto es aperture (z=2.63, FAP=0.03, rv_ok=True, FWHM=2.7 Å) — hay que juzgarlo por FAP + rv + anchura, no por z.
- LSF = 2.297 Å, 33 controles → min_resolvable_fap ≈ 0.029 (un FAP<1% estricto necesitaría ~99 controles).


## Conclusión (registrada)

**E1: veredicto `non_detection` (`no_method_passes_global_fap`).** Cifras de este objeto, resueltas de su `stage_h01_qc.json` y de `tables/halpha_detection_by_method.csv`.

- **Todos los métodos:** z 0.15–2.63, FAP 0.03–1.00 frente al umbral 0.01.
- **Pico mayor:** aperture (z=2.63, v=+18 km/s, rv_ok=True, FWHM=2.7 Å frente a una LSF de 2.297 Å).
- **Inputs:** LSF 2.297 Å, 33 controles (min_fap 0.029; ~99 para 1% estricto), rv_sys -7 km/s (literatura).
- **Downstream:** alimenta E3 (límite superior de Ṁ) y G2.
